In [4]:
import rasterio
from dist_s1.dist_plot import get_dist_s1_mpl_cmap
import matplotlib.pyplot as plt
import numpy as np
from dem_stitcher.rio_window import read_raster_from_window
from dist_s1_enumerator.mgrs_burst_data import get_mgrs_table
import pandas as pd
from pathlib import Path
from tile_mate import get_raster_from_tiles
from dem_stitcher.rio_tools import reproject_arr_to_match_profile

import earthaccess
from dist_s1_enumerator.mgrs_burst_data import get_mgrs_table
import pandas as pd
from tqdm import tqdm
from pathlib import Path

In [5]:
MGRS_TILE_ID = '22MGB' #'33MXS' #'50NPL' # '20LMM' '20MMA'
MGRS_TILE_ID = '21MWP'


In [6]:
df_mgrs = get_mgrs_table()

In [ ]:
earthaccess.login()

In [ ]:
dist_ts_dir = Path('opera_dist_data/')
ts_dir = list(dist_ts_dir.glob(f'*_{MGRS_TILE_ID}/'))[0]
ts_dir

PosixPath('opera_dist_data/builtnewalert__22MGB')

In [ ]:
status_files = sorted(list(ts_dir.rglob('OPERA*_GEN-DIST-STATUS.tif')))
status_files[:3]

[PosixPath('opera_dist_data/builtnewalert__22MGB/OPERA_L3_DIST-ALERT-S1_T22MGB_20240105T085030Z_20260205T193502Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T22MGB_20240105T085030Z_20260205T193502Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'),
 PosixPath('opera_dist_data/builtnewalert__22MGB/OPERA_L3_DIST-ALERT-S1_T22MGB_20240110T085837Z_20260205T204137Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T22MGB_20240110T085837Z_20260205T204137Z_S1A_30_v0.1_GEN-DIST-STATUS.tif'),
 PosixPath('opera_dist_data/builtnewalert__22MGB/OPERA_L3_DIST-ALERT-S1_T22MGB_20240117T085029Z_20260205T200547Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T22MGB_20240117T085029Z_20260205T200547Z_S1A_30_v0.1_GEN-DIST-STATUS.tif')]

In [ ]:
name_for_dist_hls_dir = status_files[0].parents[1].stem
name_for_dist_hls_dir

'builtnewalert__22MGB'

In [ ]:
dist_hls_dir = Path(f'dist_hls/{name_for_dist_hls_dir}')
dist_hls_dir.mkdir(exist_ok=True, parents=True)

In [ ]:
def get_acq_time(path):
    opera_id = path.stem
    tokens = opera_id.split('_')
    acq_time = pd.to_datetime(tokens[4])
    return acq_time

ts_start = get_acq_time(status_files[0])
ts_end = get_acq_time(status_files[-1])
ts_start, ts_end


(Timestamp('2024-01-05 08:50:30+0000', tz='UTC'),
 Timestamp('2024-12-30 08:50:25+0000', tz='UTC'))

In [ ]:
ts_start_mod = max(ts_start, ts_end - pd.Timedelta(days=30 * 4))
ts_start_mod, ts_start, ts_end

(Timestamp('2024-09-01 08:50:25+0000', tz='UTC'),
 Timestamp('2024-01-05 08:50:30+0000', tz='UTC'),
 Timestamp('2024-12-30 08:50:25+0000', tz='UTC'))

# Testing

In [ ]:
def get_opera_id(query_item) -> str:
    return dict(query_item.__dict__['render_dict'])['meta']['native-id']

def query_dist_hls_cmr(mgrs_tile_id: str, start_time=str(ts_start_mod.date()), stop_time=str(ts_end.date())):
    collection_short_name = 'OPERA_L3_DIST-ALERT-HLS_V1'
    mgrs_bounds = tuple(df_mgrs[df_mgrs.mgrs_tile_id == mgrs_tile_id].total_bounds)
    mgrs_bounds = tuple(float(x) for x in mgrs_bounds)
    datasets_found = earthaccess.search_data(
        short_name=collection_short_name,
        temporal=(start_time, stop_time),
        cloud_hosted=True,
        bounding_box=mgrs_bounds
    )
    datasets_found = [d for d in datasets_found if mgrs_tile_id in get_opera_id(d)]
    return datasets_found

In [ ]:
# q = july_query_dist_hls_cmr(MGRS_TILE_ID)


In [ ]:
def get_processing_time(query_item: earthaccess.DataGranule) -> pd.Timestamp:
    data = query_item.__dict__['render_dict']
    ts = pd.Timestamp(data['umm']['TemporalExtent']['RangeDateTime']['BeginningDateTime'])
    return ts

In [ ]:
#dict(qs_ordered[0].__dict__['render_dict'])

In [ ]:
# qs_ordered = sorted(q, key=get_processing_time)


In [ ]:
# earthaccess.download(q[0], '.')

# Automate

In [ ]:
def get_dist_hls_data() -> str:
    datasets = query_dist_hls_cmr(MGRS_TILE_ID)
    if datasets:
        datasets_ordered = sorted(datasets, key=get_processing_time)
        for dataset in datasets_ordered:
            opera_id = dict(dataset.__dict__['render_dict'])['meta']['native-id']
            out_dir = dist_hls_dir / opera_id
            out_dir.mkdir(exist_ok=True, parents=True)
            r = earthaccess.download(dataset, out_dir)
        return r
    else:
        return ''

In [ ]:
get_dist_hls_data()

QUEUEING TASKS | :   0%|          | 0/19 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/19 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/19 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/19 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/19 [00:00<?, ?it/s]

...

COLLECTING RESULTS | :   0%|          | 0/19 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/19 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/19 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/19 [00:00<?, ?it/s]

['dist_hls/builtnewalert__22MGB/OPERA_L3_DIST-ALERT-HLS_T22MGB_20241230T133831Z_20250101T031331Z_S2A_30_v1/OPERA_L3_DIST-ALERT-HLS_T22MGB_20241230T133831Z_20250101T031331Z_S2A_30_v1_VEG-DIST-STATUS.tif',
 'dist_hls/builtnewalert__22MGB/OPERA_L3_DIST-ALERT-HLS_T22MGB_20241230T133831Z_20250101T031331Z_S2A_30_v1/OPERA_L3_DIST-ALERT-HLS_T22MGB_20241230T133831Z_20250101T031331Z_S2A_30_v1_VEG-IND.tif',
 'dist_hls/builtnewalert__22MGB/OPERA_L3_DIST-ALERT-HLS_T22MGB_20241230T133831Z_20250101T031331Z_S2A_30_v1/OPERA_L3_DIST-ALERT-HLS_T22MGB_20241230T133831Z_20250101T031331Z_S2A_30_v1_VEG-ANOM.tif',
 'dist_hls/builtnewalert__22MGB/OPERA_L3_DIST-ALERT-HLS_T22MGB_20241230T133831Z_20250101T031331Z_S2A_30_v1/OPERA_L3_DIST-ALERT-HLS_T22MGB_20241230T133831Z_20250101T031331Z_S2A_30_v1_VEG-HIST.tif',
 'dist_hls/builtnewalert__22MGB/OPERA_L3_DIST-ALERT-HLS_T22MGB_20241230T133831Z_20250101T031331Z_S2A_30_v1/OPERA_L3_DIST-ALERT-HLS_T22MGB_20241230T133831Z_20250101T031331Z_S2A_30_v1_VEG-ANOM-MAX.tif',
 'dis